# Search Pipeline v3: Multi-vector + BM25 + CLIP + RU Index

Оценка на 500 запросах (5 EN + 5 RU на мем)

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.chdir(os.path.abspath(".." if os.path.basename(os.getcwd()) == "notebooks" else "."))

import json, numpy as np, faiss, time, re, torch
from pathlib import Path
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
from deep_translator import GoogleTranslator

# --- Данные ---
with open("data/processed/index_metadata.jsonl") as f:
    metadata = [json.loads(l) for l in f]

with open("data/processed/vqa_annotations_v2.jsonl") as f:
    all_records = [json.loads(l) for l in f]

# RU поля в metadata
fn_to_ru = {r["filename"]: r for r in all_records}
for m in metadata:
    r = fn_to_ru.get(m["filename"], {})
    m["caption_ru"] = r.get("caption_ru", "")
    m["ocr_ru"] = r.get("ocr_ru", "")


# --- FAISS ---
idx_caption = faiss.read_index("data/processed/faiss_caption.index")
idx_ocr = faiss.read_index("data/processed/faiss_ocr.index")
idx_keywords = faiss.read_index("data/processed/faiss_keywords.index")
idx_caption_ru = faiss.read_index("data/processed/faiss_caption_ru.index")
idx_image = faiss.read_index("data/processed/faiss_image.index")
idx_vqa = faiss.read_index("data/processed/faiss_vqa.index")

# --- BM25 (EN+RU) ---
def tokenize(t):
    return re.findall(r"[a-zа-яё0-9]+", t.lower())

corpus = []
for m in metadata:
    parts = [m.get("caption",""), m.get("ocr_text",""),
             m.get("caption_ru",""), m.get("ocr_ru","")]
    corpus.append(tokenize(" ".join(parts)))
bm25 = BM25Okapi(corpus)

# --- bge-m3 ---
text_model = SentenceTransformer("BAAI/bge-m3", device="cpu")

# --- CLIP ---
from transformers import CLIPModel, CLIPProcessor
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_proc = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model = clip_model.to("mps")
clip_model.eval()



## Загрузка запросов v3 (5 EN + 5 RU на мем)

In [ ]:
# --- Маппинг old → new (NSFW фильтр сдвигает индексы) ---
old_to_new = {}
ni = 0
for i, r in enumerate(all_records):
    if not r.get("is_nsfw", False):
        old_to_new[i] = ni
        ni += 1

# --- Загрузка queries_v3 ---
with open("eval/validation_set/queries_v3.json") as f:
    queries_raw = json.load(f)

# Разворачиваем: 1 мем × 5 запросов → 5 отдельных записей
valid_en = []  # (query_text, target_index)
valid_ru = []

skipped = 0
for q in queries_raw:
    nn = old_to_new.get(q["index"])
    if nn is None:
        skipped += 1
        continue
    for query_text in q["queries_en"]:
        valid_en.append({"text": query_text.strip(), "target": nn, "filename": q["filename"]})
    for query_text in q["queries_ru"]:
        valid_ru.append({"text": query_text.strip(), "target": nn, "filename": q["filename"]})

print(f"мемов: {len(queries_raw)}, пропущено (NSFW): {skipped}")
print(f"EN запросов: {len(valid_en)}, RU запросов: {len(valid_ru)}")


мемов: 100, пропущено (NSFW): 0
EN запросов: 500, RU запросов: 500


## Pre-encode всех запросов

In [ ]:
text_model = text_model.to("cpu")
clip_model = clip_model.to("cpu")

# bge-m3
en_texts = [q["text"] for q in valid_en]
en_embs = text_model.encode(en_texts, normalize_embeddings=True, batch_size=32).astype(np.float32)

ru_texts = [q["text"] for q in valid_ru]
ru_embs = text_model.encode(ru_texts, normalize_embeddings=True, batch_size=32).astype(np.float32)

# CLIP
def clip_encode_batch(texts, batch_size=32):
    all_embs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        inp = clip_proc(text=batch, return_tensors="pt", padding=True, truncation=True)
        with torch.no_grad():
            out = clip_model.get_text_features(
                input_ids=inp["input_ids"], attention_mask=inp["attention_mask"])
            if hasattr(out, "pooler_output"):
                out = out.pooler_output
            out = torch.nn.functional.normalize(out, dim=-1)
        all_embs.append(out.numpy())
    return np.concatenate(all_embs, axis=0).astype(np.float32)

clip_en = clip_encode_batch(en_texts)
clip_ru = clip_encode_batch(ru_texts)



## Оценка пайплайна

In [ ]:
def rrf(ranked_lists, k=20):
    scores = {}
    for rl in ranked_lists:
        for rank, doc_id in enumerate(rl):
            scores[doc_id] = scores.get(doc_id, 0) + 1.0 / (k + rank + 1)
    return sorted(scores, key=lambda x: -scores[x])

def evaluate(queries, embs, clip_embs=None,
             use_multi=False, use_bm25=False, use_ru=False, use_clip=False,
             tr_embs=None, tr_clip=None, use_vqa=False):
    """Считает Hit@K и MRR на списке запросов"""
    h1 = h5 = h10 = 0
    mrr = 0
    
    for i, q in enumerate(queries):
        rl = []
        qe = embs[i:i+1]
        
        # Caption (всегда)
        _, ci = idx_caption.search(qe, 50)
        rl.append(ci[0].tolist())
        
        # Multi-vector: + OCR + keywords
        if use_multi:
            _, oi = idx_ocr.search(qe, 50)
            rl.append(oi[0].tolist())
            _, ki = idx_keywords.search(qe, 50)
            rl.append(ki[0].tolist())
        
        # BM25
        if use_bm25:
            bm25_scores = bm25.get_scores(tokenize(q["text"]))
            rl.append(np.argsort(-bm25_scores)[:50].tolist())
        
        # RU index
        if use_ru:
            _, ri = idx_caption_ru.search(qe, 50)
            rl.append(ri[0].tolist())
        
        # CLIP
        if use_clip and clip_embs is not None:
            ce = clip_embs[i:i+1]
            _, ii = idx_image.search(ce, 50)
            rl.append(ii[0].tolist())
        
        if use_vqa:
            _, vi = idx_vqa.search(qe, 50)
            rl.append(vi[0].tolist())

        # RRF fusion
        ranking = rrf(rl)
        target = q["target"]
        
        if target in ranking:
            pos = ranking.index(target) + 1
            if pos <= 1: h1 += 1
            if pos <= 5: h5 += 1
            if pos <= 10: h10 += 1
            if pos <= 10: mrr += 1.0 / pos
    
    n = len(queries)
    return {"hit1": h1/n, "hit5": h5/n, "hit10": h10/n, "mrr": mrr/n}



print(f"{'конфигурация':<50} {'Hit@1':>6} {'Hit@5':>6} {'Hit@10':>7} {'MRR':>8}")


configs = [
    # EN
    ("EN: Caption only (baseline)",     valid_en, en_embs, None,    {}),
    ("EN: Multi-vector",                valid_en, en_embs, None,    {"use_multi": True}),
    ("EN: Multi+BM25",                  valid_en, en_embs, None,    {"use_multi": True, "use_bm25": True}),
    ("EN: Multi+BM25+CLIP",             valid_en, en_embs, clip_en, {"use_multi": True, "use_bm25": True, "use_clip": True}),
    ("EN: Multi+BM25+CLIP+VQA",         valid_en, en_embs, clip_en,
     {"use_multi": True, "use_bm25": True, "use_clip": True, "use_vqa": True}),
    
    # RU
    ("RU: Caption only",               valid_ru, ru_embs, None,    {}),
    ("RU: Multi-vector",                valid_ru, ru_embs, None,    {"use_multi": True}),
    ("RU: Multi+BM25",                  valid_ru, ru_embs, None,    {"use_multi": True, "use_bm25": True}),
    ("RU: Multi+BM25+RU index",         valid_ru, ru_embs, None,    {"use_multi": True, "use_bm25": True, "use_ru": True}),
    ("RU: Multi+BM25+CLIP",             valid_ru, ru_embs, clip_ru, {"use_multi": True, "use_bm25": True, "use_clip": True}),
    ("RU: Multi+BM25+CLIP+RU index",    valid_ru, ru_embs, clip_ru, {"use_multi": True, "use_bm25": True, "use_clip": True, "use_ru": True}),
    ("RU: Full+translate+VQA",           valid_ru, ru_embs, clip_ru,
     {"use_multi": True, "use_bm25": True, "use_clip": True, "use_ru": True,
      "tr_embs": ru_tr_embs, "tr_clip": ru_tr_clip, "use_vqa": True}),

    
]

results = {}
for name, queries, embs, clip_e, kwargs in configs:
    r = evaluate(queries, embs, clip_embs=clip_e, **kwargs)
    print(f"{name:<50} {r['hit1']:>6.0%} {r['hit5']:>6.0%} {r['hit10']:>7.0%} {r['mrr']:>8.4f}")
    results[name] = r

print()
print(f"Запросов: EN={len(valid_en)}, RU={len(valid_ru)} (5 на мем, {len(queries_raw)} мемов)")

# Сохраняем
with open("eval/results/pipeline_v3_results.json", "w") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
print("Сохранено в eval/results/pipeline_v3_results.json")


конфигурация                                        Hit@1  Hit@5  Hit@10      MRR


NameError: name 'ru_tr_embs' is not defined

### добавим перевод с ru на eng и будем искать запрос по двум языкам

In [ ]:
# Перевод RU на EN
from deep_translator import GoogleTranslator
translator = GoogleTranslator(source="ru", target="en")

ru_translated = []
for q in valid_ru:
    text = q["text"]
    cyr = sum(1 for c in text if "а" <= c.lower() <= "я")
    if len(text) > 0 and cyr / len(text) > 0.3:
        try:
            ru_translated.append(translator.translate(text) or text)
        except:
            ru_translated.append(text)
    else:
        ru_translated.append(text)

ru_tr_embs = text_model.encode(ru_translated, normalize_embeddings=True, batch_size=32).astype(np.float32)
ru_tr_clip = clip_encode_batch(ru_translated)


## Cross-Encoder Reranking (Stage 2)

In [ ]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("BAAI/bge-reranker-v2-m3", device="cpu")




def evaluate(queries, embs, clip_embs=None,
             use_multi=False, use_bm25=False, use_ru=False, use_clip=False,
             tr_embs=None, tr_clip=None):
    h1 = h5 = h10 = 0
    mrr = 0

    for i, q in enumerate(queries):
        rl = []
        qe = embs[i:i+1]

        _, ci = idx_caption.search(qe, 50)
        rl.append(ci[0].tolist())

        if use_multi:
            _, oi = idx_ocr.search(qe, 50)
            rl.append(oi[0].tolist())
            _, ki = idx_keywords.search(qe, 50)
            rl.append(ki[0].tolist())

        if use_bm25:
            bm25_scores = bm25.get_scores(tokenize(q["text"]))
            rl.append(np.argsort(-bm25_scores)[:50].tolist())

        if use_ru:
            _, ri = idx_caption_ru.search(qe, 50)
            rl.append(ri[0].tolist())

        if use_clip and clip_embs is not None:
            ce = clip_embs[i:i+1]
            _, ii = idx_image.search(ce, 50)
            rl.append(ii[0].tolist())

        # Translated query на EN indices
        if tr_embs is not None:
            te = tr_embs[i:i+1]
            _, tci = idx_caption.search(te, 50)
            rl.append(tci[0].tolist())

        if tr_clip is not None:
            tce = tr_clip[i:i+1]
            _, tii = idx_image.search(tce, 50)
            rl.append(tii[0].tolist())

        ranking = rrf(rl)
        target = q["target"]

        if target in ranking:
            pos = ranking.index(target) + 1
            if pos <= 1: h1 += 1
            if pos <= 5: h5 += 1
            if pos <= 10: h10 += 1
            if pos <= 10: mrr += 1.0 / pos

    n = len(queries)
    return {"hit1": h1/n, "hit5": h5/n, "hit10": h10/n, "mrr": mrr/n}


configs_rerank = [
    ("EN: Multi+BM25+CLIP + rerank",          valid_en, en_embs, clip_en,
     {"use_multi": True, "use_bm25": True, "use_clip": True}),
    ("RU: Multi+BM25+CLIP+RU idx + rerank",   valid_ru, ru_embs, clip_ru,
     {"use_multi": True, "use_bm25": True, "use_clip": True, "use_ru": True}),
     ("RU: Full + translate",                valid_ru, ru_embs, clip_ru,
     {"use_multi": True, "use_bm25": True, "use_clip": True, "use_ru": True,
      "tr_embs": ru_tr_embs, "tr_clip": ru_tr_clip})
]

print(f"{'конфиг':<50} {'Hit@1':>6} {'Hit@5':>6} {'Hit@10':>7} {'MRR':>8}")
for name, queries, embs, clip_e, kwargs in configs_rerank:
    r = evaluate(queries, embs, clip_embs=clip_e, **kwargs)
    print(f"{name:<50} {r['hit1']:>6.0%} {r['hit5']:>6.0%} {r['hit10']:>7.0%} {r['mrr']:>8.4f}")


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

конфиг                                              Hit@1  Hit@5  Hit@10      MRR
EN: Multi+BM25+CLIP + rerank                          24%    47%     56%   0.3458
RU: Multi+BM25+CLIP+RU idx + rerank                   20%    42%     48%   0.2963
RU: Full + translate                                  24%    45%     53%   0.3408


## Эксперемент с rrf fuison с весами

In [ ]:
def rrf_weighted(ranked_lists, weights=None, k=20):
    if weights is None:
        weights = [1.0] * len(ranked_lists)
    scores = {}
    for rl, w in zip(ranked_lists, weights):
        for rank, doc_id in enumerate(rl):
            scores[doc_id] = scores.get(doc_id, 0) + w / (k + rank + 1)
    return sorted(scores, key=lambda x: -scores[x])


def evaluate_weighted(queries, embs, clip_embs=None, tr_embs=None, tr_clip=None,
                      use_multi=False, use_bm25=False, use_ru=False, use_clip=False,
                      weights=None, rrf_k=20):

    h1 = h5 = h10 = 0
    mrr = 0

    for i, q in enumerate(queries):
        rl = []
        qe = embs[i:i+1]

        _, ci = idx_caption.search(qe, 50)
        rl.append(ci[0].tolist())  # 0: caption

        if use_multi:
            _, oi = idx_ocr.search(qe, 50)
            rl.append(oi[0].tolist())  # 1: ocr
            _, ki = idx_keywords.search(qe, 50)
            rl.append(ki[0].tolist())  # 2: keywords

        if use_bm25:
            bm25_scores = bm25.get_scores(tokenize(q["text"]))
            rl.append(np.argsort(-bm25_scores)[:50].tolist())  # 3: bm25

        if use_ru:
            _, ri = idx_caption_ru.search(qe, 50)
            rl.append(ri[0].tolist())  # 4: ru

        if use_clip and clip_embs is not None:
            ce = clip_embs[i:i+1]
            _, ii = idx_image.search(ce, 50)
            rl.append(ii[0].tolist())  # 5: clip

        if tr_embs is not None:
            te = tr_embs[i:i+1]
            _, tci = idx_caption.search(te, 50)
            rl.append(tci[0].tolist())  # 6: translate text

        if tr_clip is not None:
            tce = tr_clip[i:i+1]
            _, tii = idx_image.search(tce, 50)
            rl.append(tii[0].tolist())  # 7: translate clip

        ranking = rrf_weighted(rl, weights=weights, k=rrf_k)

        target = q["target"]

        if target in ranking:
            pos = ranking.index(target) + 1
            if pos <= 1: h1 += 1
            if pos <= 5: h5 += 1
            if pos <= 10: h10 += 1
            if pos <= 10: mrr += 1.0 / pos

    n = len(queries)
    return {"hit1": h1/n, "hit5": h5/n, "hit10": h10/n, "mrr": mrr/n}


# Порядок весов: [caption, ocr, keywords, bm25, ru, clip, tr_text, tr_clip]
# weight_configs = [
#     ("RU equal weights",        [1, 1, 1, 1, 1, 1, 1, 1]),
#     ("RU caption+bm25 heavy",   [2, 0.5, 0.5, 2, 1, 1, 1, 1]),
#     ("RU caption+bm25+tr heavy",[2, 0.5, 0.5, 2, 1, 0.5, 2, 0.5]),
#     ("RU no ocr/kw",            [2, 0, 0, 2, 1, 1, 2, 1]),
# ]

# k_configs = [20, 30, 40, 50, 60, 80, 100]
# print(f"{'k':<10} {'Hit@1':>6} {'Hit@5':>6} {'Hit@10':>7} {'MRR':>8}")
# for k_val in k_configs:
#     r = evaluate_weighted(
#         valid_ru, ru_embs, clip_embs=clip_ru,
#         use_multi=True, use_bm25=True, use_ru=True, use_clip=True,
#         tr_embs=ru_tr_embs, tr_clip=ru_tr_clip,
#         rrf_k=k_val
#     )
#     print(f"k={k_val:<7} {r['hit1']:>6.0%} {r['hit5']:>6.0%} {r['hit10']:>7.0%} {r['mrr']:>8.4f}")

#     r_en = evaluate_weighted(
#         valid_en, en_embs, clip_embs=clip_en,
#         use_multi=True, use_bm25=True, use_clip=True,
#         rrf_k=20
#     )
#     print(f"EN k=20: {r_en['hit1']:.0%} {r_en['hit5']:.0%} {r_en['hit10']:.0%} {r_en['mrr']:.4f}")
